# 1. 라이브러리 로드

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 2. 데이터 불러오기

In [ ]:
df = pd.read_csv('../data/marathon_results_2015.csv', index_col=0)

In [ ]:
def time_to_seconds(t):
    if pd.isna(t) or str(t).strip() in ['-', '', 'nan']:
        return None
    parts = str(t).strip().split(':')
    if len(parts) == 3:
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])
    return None

time_cols = ['5K', '10K', '15K', '20K', 'Half', '25K', '30K', '35K', '40K', 'Official Time']
for col in time_cols:
    df[col + '_s'] = df[col].apply(time_to_seconds)

df = df.dropna(subset=['Official Time_s', '5K_s', '10K_s', '30K_s', '40K_s'])
print(f"유효 데이터: {len(df):,}명")
print(f"완주 시간 평균: {df['Official Time_s'].mean()/3600:.2f}시간")
print(f"완주 시간 중앙값: {df['Official Time_s'].median()/3600:.2f}시간")

# 3. 데이터 기본 정보 확인

In [ ]:
print("--- 데이터 기본 정보 ---")
print(df.info())
print("\n--- 기초 통계량 ---")
print(df.describe())

# 4. 시각화 1: 나이대별 완주 시간 분포

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Age'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Age Distribution of Runners')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')

axes[1].scatter(df['Age'], df['Official Time_s'] / 60, alpha=0.1, s=5, color='steelblue')
axes[1].set_title('Age vs Official Finish Time')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Finish Time (minutes)')

plt.tight_layout()
plt.show()

# 5. 시각화 2: 성별에 따른 페이스 차이 (박스플롯)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='M/F', y=df['Official Time_s'] / 60, ax=axes[0], palette='Set2')
axes[0].set_title('Official Finish Time by Gender')
axes[0].set_xlabel('Gender')
axes[0].set_ylabel('Finish Time (minutes)')

df['AgeGroup'] = pd.cut(df['Age'], bins=[17, 29, 39, 49, 59, 82],
                        labels=['18-29', '30-39', '40-49', '50-59', '60+'])
age_mean = df.groupby('AgeGroup', observed=True)['Official Time_s'].mean() / 60
age_mean.plot(kind='bar', ax=axes[1], color='steelblue', rot=0)
axes[1].set_title('Mean Finish Time by Age Group')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Mean Finish Time (minutes)')

plt.tight_layout()
plt.show()

# 6. 시각화 3: 기온과 완주 시간의 상관관계 (산점도)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

corr_features = ['Age', 'Official Time_s', '5K_s', '10K_s', '30K_s']
corr_matrix = df[corr_features].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[0])
axes[0].set_title('Correlation Matrix (Key Features)')

top_countries = df['Country'].value_counts().head(8)
top_countries.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Top 8 Countries by Participant Count')
axes[1].set_xlabel('Number of Runners')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print("\n--- 완주 시간 기초 통계 ---")
print(df['Official Time_s'].describe().apply(lambda x: f"{int(x//3600)}h{int((x%3600)//60):02d}m"))